# Prototipo Deuna FINS: Motor de Reglas Crediticias, Tu nivel deuna y Gamificación

## Introducción

El presente cuaderno documenta de forma sistemática la lógica de negocio implementada en el prototipo **FINS** (Financial Inclusion & Nanocredit Simulator), una aplicación orientada al ecosistema **Deuna** que modela productos de microfinanzas digital para el mercado ecuatoriano. El sistema aborda dos verticales complementarias:

1. **Dame un Chance (Tu nivel deuna):** nanocréditos de consumo ($3.50, $7.50 y $10.00) con **plazo de 7 días** y tarifa de **$0.25** (gastos aplicativos), condicionados al puntaje alternativo de confianza (*Tu nivel deuna*).
2. **Deuna Negocio (Veci):** microcréditos comerciales ($50–$300) con cupo dinámico estimado a partir de ventas QR simuladas, amortización regulatoria (28% nominal anual) y retención automática sobre cobros.

La implementación de referencia reside en `src/app/lib/creditRules.ts` (reglas puras) y `src/app/context/AppContext.tsx` (orquestación de estado y flujos transaccionales). Este notebook **replica y valida** dichas reglas en Python para fines de reproducibilidad académica, auditoría y experimentación con *jupyter-ai*.

**Objetivos del documento:**
- Formalizar la metodología de cálculo crediticio y de gamificación.
- Permitir la ejecución independiente de escenarios sin desplegar la interfaz React.
- Establecer resultados esperados verificables por tramos de *Tu nivel deuna*, límites de Deuna Coins y perfiles de ventas.

# Metodología — Dependencias y Configuración del Entorno

Se importan las bibliotecas estándar para tipado, enumeraciones, generación de datos sintéticos y visualización tabular. La semilla aleatoria garantiza reproducibilidad en las simulaciones de ventas trimestrales, alineadas con `simulateThreeMonthSalesAverage` del código fuente TypeScript.

# Metodología — Plazo de 7 días, tarifa $0.25 y economía de Coins

Constantes alineadas con `creditRules.ts` y `coinLimits.ts`:

- **Nanocrédito:** `CHANCE_LOAN_TERM_DAYS = 7`, gastos aplicativos **$0.25**, pago temprano (días 1–5) vs. vencimiento (día 7).
- **Coins:** límites por fuente y tope diario 30; **bono ahorro:** saldo ≥ $5 durante 24 h continuas → +1 coin (máx. 1/día).
- **Cofre:** cooldown 12 h. **Ruleta:** +1, +3 o sin premio.

In [ ]:
CHANCE_LOAN_TERM_DAYS = 7
CHANCE_PLATFORM_FEE_USD = 0.25
MIN_BALANCE_FOR_SAVINGS_COIN = 5
BALANCE_HOLD_HOURS = 24

COIN_LIMITS_DOC = {
    "transfer": 6,
    "recharge": 8,
    "qr_pay": 10,
    "chest": 4,
    "ruleta": 12,
    "balance_hold": 1,
    "total_daily_cap": 30,
}

def chance_loan_row(amount: float) -> dict:
    return {
        "capital": amount,
        "gastos_aplicativos": CHANCE_PLATFORM_FEE_USD,
        "total": amount + CHANCE_PLATFORM_FEE_USD,
        "plazo_dias": CHANCE_LOAN_TERM_DAYS,
    }

[chance_loan_row(a) for a in (3.5, 7.5, 10.0)]

In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from enum import Enum
from typing import Literal

import pandas as pd

# Reproducibilidad en simulaciones estocásticas (equivalente a jitter en creditRules.ts)
random.seed(42)

print("Entorno configurado correctamente.")
print(f"pandas {pd.__version__}")

### Resultados Esperados — Configuración

Al ejecutar la celda anterior se debe observar la confirmación del entorno y la versión de `pandas`. No se producen artefactos gráficos; su función es habilitar las secciones metodológicas subsiguientes.

# Metodología — Constantes de Dominio y Catálogo de Productos

Las constantes definen el espacio de decisión del motor de reglas. Los montos de **Dame un Chance** están fijados en el arreglo `CHANCE_AMOUNTS`, coherente con la regulación de usura del BCE (0% interés nominal en capital + tarifa de plataforma en la UX). La ruleta de beneficios expone ocho segmentos equiprobables cuyos premios alimentan variables conductuales del *Tu nivel deuna* en el modelo de IA descrito en la propuesta del proyecto.

In [ ]:
CHANCE_AMOUNTS: tuple[float, ...] = (3.5, 7.5, 10.0)
ChanceAmount = Literal[3.5, 7.5, 10.0]

class ChestTier(str, Enum):
    BRONCE = "bronce"
    PLATA = "plata"
    DIAMANTE = "diamante"

@dataclass(frozen=True)
class RuletaPrize:
    label: str
    type: Literal["coins", "cosmetic", "xp"]
    value: float | str

RULETA_PRIZES: tuple[RuletaPrize, ...] = (
    RuletaPrize("+$2.00 Coins", "coins", 2),
    RuletaPrize("Borde Arcoíris", "cosmetic", "border_rainbow"),
    RuletaPrize("Corona de Elite", "cosmetic", "accessory_crown"),
    RuletaPrize("Sombrero Vaquero", "cosmetic", "accessory_cowboy"),
    RuletaPrize("+$5.00 Coins", "coins", 5),
    RuletaPrize("+20 XP", "xp", 20),
    RuletaPrize("Borde de Fuego", "cosmetic", "border_fire"),
    RuletaPrize("+50 XP Bonus", "xp", 50),
)

RULETA_SEGMENT_DEG = 360 / len(RULETA_PRIZES)

pd.DataFrame(
    [{"segmento": i, "premio": p.label, "tipo": p.type} for i, p in enumerate(RULETA_PRIZES)]
)

### Resultados Esperados — Constantes

Se despliega una tabla de ocho filas (segmentos 0–7) con etiquetas de premios. Este inventario debe coincidir con `RULETA_PRIZES` en `creditRules.ts` y garantiza trazabilidad entre frontend y análisis offline.

# Metodología — Tu nivel deuna y Elegibilidad Crediticia

El **Tu nivel deuna** (rango operativo 10–100 en `AppContext`) sustituye reportes tradicionales de buró para usuarios informalizados. Las funciones siguientes implementan:

- Clasificación en tiers (*Bronce*, *Plata*, *Diamante*).
- Umbral mínimo de elegibilidad (Tu nivel deuna > 30 pts; variable `pulsoScore` en código).
- Monto máximo de nanocrédito desbloqueado por tramo.
- Asignación del tier de cofre inteligente (umbrales 56 y 76).

Esta segmentación es crítica porque acopla riesgo crediticio con recompensas gamificadas, cerrando el ciclo virtuoso descrito en la propuesta de valor del proyecto.

In [ ]:
def get_pulso_tier_label(score: float) -> str:
    if score <= 55:
        return "Bronce"
    if score <= 75:
        return "Plata"
    return "Diamante"


def can_request_chance(pulso_score: float) -> bool:
    return pulso_score > 30


def get_max_chance_amount(pulso_score: float) -> float:
    if pulso_score <= 30:
        return 0.0
    if pulso_score <= 55:
        return 3.5
    if pulso_score <= 75:
        return 7.5
    return 10.0


def is_chance_amount_unlocked(amount: float, pulso_score: float) -> bool:
    max_amt = get_max_chance_amount(pulso_score)
    if max_amt == 0:
        return False
    return amount <= max_amt


def get_chest_tier_for_pulso(pulso_score: float) -> ChestTier:
    if pulso_score < 56:
        return ChestTier.BRONCE
    if pulso_score < 76:
        return ChestTier.PLATA
    return ChestTier.DIAMANTE


pulso_grid = pd.DataFrame({"pulso_score": range(10, 101, 5)})
pulso_grid["tier_label"] = pulso_grid["pulso_score"].map(get_pulso_tier_label)
pulso_grid["max_chance_usd"] = pulso_grid["pulso_score"].map(get_max_chance_amount)
pulso_grid["elegible"] = pulso_grid["pulso_score"].map(can_request_chance)
pulso_grid["cofre"] = pulso_grid["pulso_score"].map(lambda s: get_chest_tier_for_pulso(s).value)

pulso_grid

### Resultados Esperados — Tu nivel deuna

La tabla evidencia discontinuidades en los umbrales 31, 56, 76 y 100:
- Con *Tu nivel deuna* ≤ 30, `max_chance_usd = 0` y `elegible = False`.
- Entre 31 y 55, solo $3.50 está habilitado.
- Entre 56 y 75, hasta $7.50.
- A partir de 76, el cupo personal alcanza $10.00 y el cofre asciende a *diamante*.

Estos cortes deben validarse contra la interfaz de solicitud de *Dame un Chance* en el prototipo React.

# Metodología — Crédito Veci: Cupo IA y Simulación de Ventas QR

Para el segmento **Veci** (microcomercio), el límite de crédito se deriva del promedio de ventas mensuales por QR. La función `compute_veci_credit_limit` aplica una política conservadora: el 50% del flujo promedio, redondeado a múltiplos de $5, acotado entre **$50** y **$300**. La simulación trimestral introduce variación estocástica (±12%) para emular incertidumbre de series temporales antes de desplegar modelos LSTM/Prophet en producción.

In [ ]:
def compute_veci_credit_limit(monthly_sales_average: float) -> int:
    raw = round((monthly_sales_average * 0.5) / 5) * 5
    return int(min(300, max(50, raw)))


def simulate_three_month_sales_average(seed_base: float = 520.0) -> int:
    def jitter() -> float:
        return 0.88 + random.random() * 0.22

    m1 = round(seed_base * jitter())
    m2 = round(seed_base * jitter())
    m3 = round(seed_base * jitter())
    return round((m1 + m2 + m3) / 3)


N_SIM = 20
veci_sim = pd.DataFrame(
    {
        "simulacion": range(1, N_SIM + 1),
        "ventas_promedio_3m": [simulate_three_month_sales_average(520) for _ in range(N_SIM)],
    }
)
veci_sim["cupo_ia_usd"] = veci_sim["ventas_promedio_3m"].map(compute_veci_credit_limit)
veci_sim.describe().round(2)

### Resultados Esperados — Cupo IA Veci

Las estadísticas descriptivas deben mostrar `cupo_ia_usd` siempre dentro de [50, 300]. Con `seed_base = 520`, el promedio trimestral simulado ronda $520±, produciendo cupos cercanos a **$130** (50% de ~260 redondeado). Variaciones entre ejecuciones reflejan el *jitter* aleatorio; fijar `random.seed(42)` estabiliza corridas sucesivas para informes reproducibles.

# Metodología — Ingeniería Financiera: Costo Total y Retención QR

El cálculo `calculate_veci_loan_totals` materializa el cumplimiento regulatorio ecuatoriano:

- **Interés:** tasa nominal anual del **28%**, prorrateada por días (`months × 30 / 360`).
- **Seguro de desgravamen:** **0.1% mensual** sobre capital (`0.001 × months`).

La retención automática sobre ventas QR adopta **8%** por defecto y **6%** cuando existe historial positivo (`consecutive_good_payments ≥ 1`) o *Tu nivel deuna* ≥ 76, incentivando comportamiento responsable sin ahogar la liquidez del comercio.

In [ ]:
@dataclass
class VeciLoanTotals:
    interest: float
    insurance: float
    total: float


def calculate_veci_loan_totals(amount: float, months: int) -> VeciLoanTotals:
    interest = amount * 0.28 * ((months * 30) / 360)
    insurance = amount * 0.001 * months
    return VeciLoanTotals(interest=interest, insurance=insurance, total=amount + interest + insurance)


def get_veci_retention_rate(pulso_score: float, consecutive_good_payments: int) -> float:
    if consecutive_good_payments >= 1 or pulso_score >= 76:
        return 0.06
    return 0.08


loan_matrix = []
for amount in [50, 100, 200, 300]:
    for months in [1, 2, 3]:
        t = calculate_veci_loan_totals(amount, months)
        loan_matrix.append(
            {
                "capital": amount,
                "plazo_meses": months,
                "interes": round(t.interest, 2),
                "seguro": round(t.insurance, 2),
                "total_a_pagar": round(t.total, 2),
            }
        )

loan_df = pd.DataFrame(loan_matrix)
loan_df["retencion_8pct"] = (loan_df["total_a_pagar"] / loan_df["capital"]).round(3)
loan_df

### Resultados Esperados — Amortización Veci

Para un préstamo de **$100 a 1 mes**, el interés debe ser ≈ **$2.33** (`100 × 0.28 × 30/360`) y el seguro **$0.10**, totalizando **$102.43**. La columna auxiliar de retención ilustra cuántas ventas equivalentes requeriría amortizar al 8%; en producción, la retención se aplica transacción a transacción mediante `simulateVeciQRSale` en `AppContext.tsx`.

# Metodología — Simulación de Cobro QR con Retención Automática

Esta sección modela el flujo central de **amortización por retención de ventas**, replicando la lógica de `simulateVeciQRSale`: retención proporcional al saldo activo y acreditación neta al comerciante. El objetivo es cuantificar cuántas ventas diarias de magnitud fija liquidan un crédito bajo tasas del 6% u 8%.

In [ ]:
def simulate_qr_sale_waterfall(
    sale_amount: float,
    remaining_debt: float,
    retention_rate: float,
) -> dict:
    retention_amt = sale_amount * retention_rate if remaining_debt > 0 else 0.0
    retention_amt = min(retention_amt, remaining_debt)
    net_credited = sale_amount - retention_amt
    new_remaining = max(0.0, remaining_debt - retention_amt)
    return {
        "venta_bruta": sale_amount,
        "retencion": round(retention_amt, 2),
        "neto_acreditado": round(net_credited, 2),
        "saldo_restante": round(new_remaining, 2),
        "liquidado": new_remaining == 0,
    }


CAPITAL = 100.0
MESES = 2
totals = calculate_veci_loan_totals(CAPITAL, MESES)
remaining = totals.total
ret_rate = 0.08
DAILY_SALE = 25.0

history = []
day = 0
while remaining > 0 and day < 60:
    day += 1
    step = simulate_qr_sale_waterfall(DAILY_SALE, remaining, ret_rate)
    step["dia"] = day
    history.append(step)
    remaining = step["saldo_restante"]

amort_df = pd.DataFrame(history)
print(f"Crédito Veci: ${CAPITAL} a {MESES} meses | Total adeudado: ${totals.total:.2f}")
print(f"Días hasta liquidación (ventas diarias ${DAILY_SALE}, retención {ret_rate*100:.0f}%): {day}")
amort_df.tail(5)

### Resultados Esperados — Waterfall QR

La simulación debe converger a `saldo_restante = 0` en un número finito de días. Con retención del 8% sobre ventas de $25, cada día se destinan **$2.00** a deuda. Al finalizar, `liquidado = True` en la última fila. Reducir la retención al 6% (perfil premium) alarga el plazo de amortización pero mejora la liquidez diaria del *Veci*.

# Metodología — Dame un Chance: Estructura de Costo en el Prototipo

El nanocrédito personal liquida en **7 días calendario**. Costo: **capital + $0.25** (gastos aplicativos, 0% interés sobre principal), vía `getChanceLoanTotal` y `AppContext.requestSalvavidas`. Cobro automático en recargas; pago temprano simulado días 1–5, al vencimiento día 7.

In [ ]:
def salvavidas_totals_demo(amount: float) -> dict:
    """Réplica de requestSalvavidas en AppContext.tsx (demo)."""
    return {"capital": amount, "platform_fee": 0.25, "total": amount + 0.25, "plazo_dias": 7}


def salvavidas_totals_ux(amount: float) -> dict:
    """Modelo comunicado al usuario (propuesta Deuna)."""
    platform_fee = 0.25
    return {"capital": amount, "gastos_aplicativos": platform_fee, "total_ux": amount + platform_fee, "plazo_dias": 7}


chance_rows = []
for amt in CHANCE_AMOUNTS:
    demo = salvavidas_totals_demo(amt)
    ux = salvavidas_totals_ux(amt)
    chance_rows.append({**demo, **ux})

pd.DataFrame(chance_rows)

### Resultados Esperados — Dame un Chance

Para $3.50 el total es **$3.75**; para $10.00 es **$10.25**. Plazo: **7 días**. La tabla debe reflejar capital, fee $0.25 y total en todos los montos de `CHANCE_AMOUNTS`.

# Metodología — Gamificación: Ruleta y Rotación Angular

La función `get_rotation_for_prize_index` calcula el ángulo de parada de la ruleta alineando el centro del segmento ganador bajo el puntero, más cinco vueltas completas por efecto visual. Aunque la selección del premio es uniforme en `spinRuleta`, esta geometría garantiza coherencia entre índice lógico y animación CSS en el frontend.

In [ ]:
def get_rotation_for_prize_index(prize_index: int, current_rotation: float) -> float:
    segment_center = prize_index * RULETA_SEGMENT_DEG + RULETA_SEGMENT_DEG / 2
    align_under_pointer = 360 - segment_center
    return current_rotation + 5 * 360 + align_under_pointer


rotations = pd.DataFrame(
    {
        "prize_index": range(len(RULETA_PRIZES)),
        "premio": [p.label for p in RULETA_PRIZES],
        "rotacion_final_grados": [
            get_rotation_for_prize_index(i, 0) for i in range(len(RULETA_PRIZES))
        ],
    }
)
rotations

### Resultados Esperados — Ruleta

Cada índice produce un ángulo > 1800° (cinco vueltas + alineación). Los valores difieren entre segmentos adyacentes en pasos de 45° (`RULETA_SEGMENT_DEG`), verificando la partición uniforme del círculo.

# Análisis de Resultados — Escenario Integrado de Usuario

Se consolida un caso nominal basado en el estado inicial del prototipo (`pulsoScore = 48`, sin crédito Veci activo) y un caso de comercio maduro con ventas proyectadas elevadas. El análisis cuantifica elegibilidad, cupo y tasa de retención en un único marco tabular.

In [ ]:
def build_user_profile(
    nombre: str,
    pulso: float,
    ventas_seed: float,
    pagos_buenos: int,
) -> dict:
    forecast = simulate_three_month_sales_average(ventas_seed)
    return {
        "perfil": nombre,
        "pulso_score": pulso,
        "tier_pulso": get_pulso_tier_label(pulso),
        "max_chance_usd": get_max_chance_amount(pulso),
        "cofre": get_chest_tier_for_pulso(pulso).value,
        "ventas_forecast": forecast,
        "cupo_veci_usd": compute_veci_credit_limit(forecast),
        "retencion_qr": f"{get_veci_retention_rate(pulso, pagos_buenos) * 100:.0f}%",
    }


profiles = pd.DataFrame(
    [
        build_user_profile("Usuario inicial (demo)", 48, 0, 0),
        build_user_profile("Veci en crecimiento", 72, 800, 0),
        build_user_profile("Veci premium", 82, 1200, 2),
    ]
)
profiles

### Resultados Esperados — Perfiles Integrados

| Perfil | Hallazgo esperado |
|--------|-------------------|
| Usuario inicial | `max_chance_usd = 3.5`, cofre *bronce*, cupo Veci mínimo ($50) por ventas forecast bajas |
| Veci en crecimiento | `max_chance_usd = 7.5`, cupo intermedio (~$200), retención 8% |
| Veci premium | `max_chance_usd = 10`, cofre *diamante*, cupo cercano a $300, retención 6% |

Estos resultados validan la coherencia transversal entre gamificación, nanocrédito y condiciones comerciales.

# Conclusiones

El prototipo **FINS** implementa un motor de reglas determinista que articula inclusión financiera, cumplimiento regulatorio ecuatoriano y mecánicas de engagement. Las contribuciones principales documentadas en este cuaderno son:

1. **Segmentación por Tu nivel deuna:** traducción directa del riesgo conductual en límites de nanocrédito y recompensas (cofres, cosméticos, retención preferencial).
2. **Cupo IA para Veci:** política transparente (`50%` del flujo, cap $300) preparada para ser sustituida por predicciones LSTM/Prophet sin alterar la interfaz de negocio.
3. **Amortización transaccional:** retención QR como innovación de repago que vincula TPV y recuperación de cartera, simulable offline mediante waterfall de ventas.
4. **Trazabilidad académica:** la migración TypeScript → Python permite auditorías, pruebas de hipótesis y extensión con modelos de ML en celdas posteriores.

**Deuna Coins:** límites en `coinLimits.ts` (transfer 6/d, recarga 8, QR 10, cofre 4, ruleta 12, **ahorro ≥$5×24h → 1 coin/d**, tope 30). Ruleta: +1/+3 o vacío. Cofre: cada 12 h.

**Limitaciones:** ventas Veci sintéticas; IA producción (XGBoost/LSTM) en `propuesta_deuna_ia.md`.

**Trabajo futuro:** integrar series reales de cobros QR, calibrar el factor de cupo frente a volatilidad sectorial y exportar métricas de cobertura (PD, LGD) hacia un servicio de scoring batch consumible por la aplicación React.